# 🪝 Node-Style Middleware in LangChain

---

## 📖 What is this notebook?

This is a **reference tutorial** for node-style middleware — the hooks that fire **before** or **after** specific nodes in the LangChain agent loop. If you're coming from web development, think Express/Django middleware, but for LLM agent graphs.

---

## 🧠 The Big Picture: Why Middleware Exists

An LLM agent is a **loop**:

```
User message → LLM thinks → (maybe calls tools) → tool results → LLM thinks again → ...
```

Every iteration **appends** to `messages[]`. After enough turns, three problems hit:

| Problem | Why it hurts |
|---|---|
| 🪙 **Context window overflow** | Even 128k-token models run out in long sessions |
| 💸 **Cost explosion** | Every token in the prompt is billed on every LLM call |
| 🌀 **Quality degradation** | "Lost in the middle" — LLMs reason worse with bloated prompts |

**Middleware** lets you intercept and transform the state at specific points in the loop — **before** the problem reaches the LLM.

---

## 🔄 The Agent Loop & Where Middleware Hooks In

![Agent Loop](resources/agent_diagram_small_arrows_FIXED.svg)

The agent graph has two main **nodes**:
1. **Agent node** — where the LLM is called
2. **Tool node** — where tools are executed

Node-style middleware hooks fire at the **transitions** between these nodes:

```
                    ┌─────────────────┐
  @before_agent ──▶ │   Agent Node    │ ──▶ @after_agent
                    │  (LLM call)     │
                    └────────┬────────┘
                             │ tool_use?
                             ▼
                    ┌─────────────────┐
  @before_tools ──▶ │   Tool Node     │ ──▶ @after_tools
                    │  (execute tools) │
                    └────────┬────────┘
                             │ tool_result
                             ▼
                       (back to Agent)
```

---

## 🪝 The Four Node-Style Hooks

| Hook | Decorator | When it fires | Receives | Returns |
|---|---|---|---|---|
| Before Agent | `@before_agent` | Before each LLM call | `AgentState` + `Runtime` | `dict` of state updates (or `None`) |
| After Agent | `@after_agent` | After LLM responds | `AgentState` + `Runtime` | `dict` of state updates (or `None`) |
| Before Tools | `@before_tools` | Before tools execute | `AgentState` + `Runtime` | `dict` of state updates (or `None`) |
| After Tools | `@after_tools` | After tools return | `AgentState` + `Runtime` | `dict` of state updates (or `None`) |

> 💡 **Key rule**: if the middleware returns `None`, the state passes through unchanged. If it returns a dict, those fields are merged into the state.

---

## 🔑 The Signature Pattern

All four hooks follow the same pattern:

```python
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from typing import Any

@before_agent
def my_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    # state["messages"] — the conversation history
    # runtime.context   — read-only context (if context_schema was set)
    # runtime.state     — the full mutable state
    
    # Return a dict to modify state, or None to pass through
    return {"messages": modified_messages}
```

Then pass it to `create_agent()`:

```python
agent = create_agent(
    model="gpt-5-nano",
    middleware=[my_middleware],  # ← list of middleware
)
```

In [ ]:
from dotenv import load_dotenv

load_dotenv()

---

## 📝 Example 1: `SummarizationMiddleware` (built-in)

### 🤔 What is it?

A built-in `@before_agent` middleware that automatically **summarizes old messages** when the conversation gets too long.

Instead of:
```
[msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]  ← growing forever
```

You get:
```
[summary_of_1_through_5, msg6, msg7, msg8]  ← bounded size
```

The agent still "remembers" early turns via the summary, but the token count stays manageable.

> 🎯 Think of it as **automatic RAG over your own conversation history**.

### ⚙️ Parameters

| Parameter | What it does | Example |
|---|---|---|
| `model` | The LLM used for summarization (can be cheap/fast) | `"gpt-4o-mini"` |
| `trigger` | When to summarize | `("tokens", 100)` or `("messages", 10)` |
| `keep` | How many recent messages to keep untouched | `("messages", 1)` |

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",       # cheap model just for summarization
            trigger=("tokens", 100),    # summarize when messages exceed 100 tokens
            keep=("messages", 1)        # always keep the most recent 1 message
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

# Simulate a long conversation — the middleware will summarize the older messages
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were the new president, how would you respond?"),
    ]},
    {"configurable": {"thread_id": "1"}}
)

# The first message should now be a summary, not the original first message
print("First message (should be a summary):")
print(response["messages"][0].content[:200] + "...")

---

## ✂️ Example 2: Custom `@before_agent` — Trim Tool Messages

### 🤔 Why strip ToolMessages?

Tool results often contain **noisy internal data** (raw JSON, debug info, verbose API responses) that the LLM doesn't need to see again on subsequent turns. Stripping them:

- 🪙 Saves tokens
- 🎯 Reduces distraction
- 🔒 Prevents leaking internal data back into the prompt

### ⚙️ How `RemoveMessage` works

LangGraph's `RemoveMessage` is a special message type that, when returned in a state update, **deletes** the message with the matching `id` from the conversation history. It's the clean way to remove messages without mutating the list directly.

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent

@before_agent
def trim_tool_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all ToolMessages from the conversation history before each LLM call."""
    messages = state["messages"]
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    if not tool_messages:
        return None  # nothing to remove — pass through
    
    # Return RemoveMessage for each ToolMessage we want to delete
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent_trimmed = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_tool_messages],
)

In [ ]:
# Conversation with noisy ToolMessages mixed in
response = agent_trimmed.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping...", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v ... greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
    ]},
    {"configurable": {"thread_id": "2"}}
)

# The LLM won't see the ToolMessages — it can't answer about temperature
print(response["messages"][-1].content)

---

## 🧩 When to Use Each Hook

| Scenario | Hook | Why |
|---|---|---|
| Summarize long conversations | `@before_agent` | Compress history before LLM sees it |
| Strip noisy tool results | `@before_agent` | Remove clutter before next LLM call |
| Log every LLM response | `@after_agent` | Capture what the LLM said |
| Validate tool calls before execution | `@before_tools` | Block dangerous operations |
| Cache tool results | `@after_tools` | Avoid re-running expensive tools |
| Inject system context per-turn | `@before_agent` | Add time-sensitive info to each call |
| Redact PII from responses | `@after_agent` | Clean output before user sees it |

---

## ⚠️ Important Gotchas

1. **Middleware order matters** — they run in the order you list them in `middleware=[]`. Put summarization before trimming if you want both.

2. **Middleware modifies what the LLM *sees*, not what *happened*** — the original messages existed. Middleware decides which version of history the LLM gets.

3. **Checkpointer interaction** — middleware runs *before* the checkpoint is saved. So the trimmed/summarized version is what gets persisted. If you summarize, the original messages are gone from the checkpoint.

4. **Context is read-only** — middleware can read `runtime.context` but cannot modify it. Only `state` (via return dict) can be changed.

5. **Return `None` to pass through** — if your middleware has nothing to do on a particular turn, return `None` (not an empty dict).

---

## 🗺️ Where Node-Style Fits in the Middleware Family

| Style | Decorators | What they control |
|---|---|---|
| 🪝 **Node-style** (this notebook) | `@before_agent`, `@after_agent`, `@before_tools`, `@after_tools` | State transformations at graph node boundaries |
| 🔀 **Wrap-style** | `@wrap_model_call`, `@wrap_tool_call` | The actual LLM/tool invocation (request → handler → response) |
| ⚙️ **Config-style** | `@dynamic_prompt`, `@hook_config` | Prompt injection and agent configuration |
| 📦 **Built-in classes** | `SummarizationMiddleware`, `PIIMiddleware`, `HumanInTheLoopMiddleware`, etc. | Pre-built solutions for common patterns |

See `dynamic_middleware.ipynb` for wrap-style and config-style middleware.